# Step 2: Experiment Execution

This notebook runs the nonlinear Granger causality test with **N random initializations**
using the best hyperparameters found in Step 1.

**Pipeline**: 01_hyperparameter_search → `02_run_experiment` → 03_results_analysis

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import src  # Sets TF_USE_LEGACY_KERAS

import numpy as np
import pickle
import time
import gc
from datetime import datetime

from src.config import get_direction_labels, validate_map, validate_direction, validate_architecture
from src.data import generate_data, normalize_data, split_data
from src.causality import run_causality_test

print("Imports successful.")

## Configuration

Set the hyperparameters identified in Step 1.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

CHAOTIC_MAP = "henon"
CAUSALITY_DIRECTION = "Y_to_X"
NN_ARCHITECTURE = "GRU"

# Best hyperparameters (from Step 1)
NN_NEURONS = [100]
MAX_LAG = 5
BATCH_SIZE_NN = 16
EPOCHS = 50
LEARNING_RATE = 0.0001

# Time series
N_ITERATIONS = 500

# Experiment
N_RUNS = 100  # WARNING: 100 runs can take 12+ hours
BATCH_SIZE_RUNS = 10

# Data splits
TRAIN_RATIO = 0.7
VAL_RATIO = 0.2

# Output
OUTPUT_DIR = "../results/experiments/"

# =============================================================================
validate_map(CHAOTIC_MAP)
validate_direction(CAUSALITY_DIRECTION)
validate_architecture(NN_ARCHITECTURE)

labels = get_direction_labels(CAUSALITY_DIRECTION)

EXPERIMENT_NAME = f"results_{CHAOTIC_MAP}_{NN_ARCHITECTURE}_{CAUSALITY_DIRECTION}"

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Runs: {N_RUNS} | Batches: {N_RUNS // BATCH_SIZE_RUNS}")

## Data Generation

In [ ]:
data = generate_data(CHAOTIC_MAP, CAUSALITY_DIRECTION, N_ITERATIONS)
data = normalize_data(data)
data_train, data_val, data_test = split_data(data, TRAIN_RATIO, VAL_RATIO)

print(f"Data shape: {data.shape}")
print(f"Train: {data_train.shape} | Val: {data_val.shape} | Test: {data_test.shape}")
print(f"Normalized range: [{data.min():.4f}, {data.max():.4f}]")

## Run Experiment

Runs the causality test N times. Intermediate checkpoints are saved after each batch.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

n_batches = N_RUNS // BATCH_SIZE_RUNS
if N_RUNS % BATCH_SIZE_RUNS > 0:
    n_batches += 1

p_values = np.zeros(N_RUNS)
test_statistics = np.zeros(N_RUNS)
rss_restricted = np.zeros(N_RUNS)
rss_full = np.zeros(N_RUNS)
cohens_d = np.zeros(N_RUNS)

print(f"Starting experiment: {EXPERIMENT_NAME}")
print(f"Total runs: {N_RUNS} | Batches: {n_batches} | Runs per batch: {BATCH_SIZE_RUNS}")
print("=" * 60)

start_time = time.time()

for batch in range(n_batches):
    start_idx = batch * BATCH_SIZE_RUNS
    end_idx = min(start_idx + BATCH_SIZE_RUNS, N_RUNS)

    print(f"\nBatch {batch + 1}/{n_batches} (runs {start_idx + 1} to {end_idx})")

    for i in range(start_idx, end_idx):
        print(f"  Run {i + 1}/{N_RUNS}...", end=" ")

        metrics, error_msg = run_causality_test(
            data_train, data_val, data_test,
            lag=MAX_LAG, architecture=NN_ARCHITECTURE,
            neurons=NN_NEURONS[0], epochs=EPOCHS,
            learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE_NN,
        )

        if metrics is not None:
            p_values[i] = metrics["p_value"]
            test_statistics[i] = metrics["test_statistic"]
            rss_restricted[i] = metrics["RSS_restricted"]
            rss_full[i] = metrics["RSS_full"]
            cohens_d[i] = metrics["cohens_d"]
            print(f"p={p_values[i]:.4f}, d={cohens_d[i]:.4f}")
        else:
            p_values[i] = test_statistics[i] = np.nan
            rss_restricted[i] = rss_full[i] = cohens_d[i] = np.nan
            print(f"ERROR: {error_msg}")

        gc.collect()

    # Save checkpoint
    cp_path = os.path.join(OUTPUT_DIR, f"{EXPERIMENT_NAME}_batch_{batch + 1}.pkl")
    cp = {
        "config": {"chaotic_map": CHAOTIC_MAP, "causality_direction": CAUSALITY_DIRECTION,
                   "nn_architecture": NN_ARCHITECTURE},
        f"lag_{MAX_LAG}": {
            "p_value": p_values[:end_idx].copy(),
            "test_statistic": test_statistics[:end_idx].copy(),
            "RSS_restricted": rss_restricted[:end_idx].copy(),
            "RSS_full": rss_full[:end_idx].copy(),
            "cohens_d": cohens_d[:end_idx].copy(),
        },
    }
    with open(cp_path, "wb") as f:
        pickle.dump(cp, f)
    print(f"  → Checkpoint saved: {cp_path}")

elapsed = time.time() - start_time
print(f"\nExperiment complete in {elapsed:.1f}s ({elapsed / 60:.1f} min)")

## Save Final Results

In [ ]:
final = {
    "config": {
        "chaotic_map": CHAOTIC_MAP,
        "causality_direction": CAUSALITY_DIRECTION,
        "direction_str": labels["direction_str"],
        "target_variable": labels["target_label"],
        "cause_variable": labels["cause_label"],
        "nn_architecture": NN_ARCHITECTURE,
        "nn_neurons": NN_NEURONS,
        "max_lag": MAX_LAG,
        "batch_size_nn": BATCH_SIZE_NN,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "n_iterations": N_ITERATIONS,
        "n_runs": N_RUNS,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "timestamp": datetime.now().isoformat(),
        "execution_time_seconds": elapsed,
    },
    f"lag_{MAX_LAG}": {
        "p_value": p_values,
        "test_statistic": test_statistics,
        "RSS_restricted": rss_restricted,
        "RSS_full": rss_full,
        "cohens_d": cohens_d,
    },
}

final_path = os.path.join(OUTPUT_DIR, f"{EXPERIMENT_NAME}_final.pkl")
with open(final_path, "wb") as f:
    pickle.dump(final, f)

# Cleanup checkpoints
for b in range(1, n_batches + 1):
    cp = os.path.join(OUTPUT_DIR, f"{EXPERIMENT_NAME}_batch_{b}.pkl")
    try:
        os.remove(cp)
    except OSError:
        pass

print(f"Final results saved: {final_path}")
print(f"\nSummary (lag_{MAX_LAG}):")
valid_p = p_values[~np.isnan(p_values)]
print(f"  p-value: {np.mean(valid_p):.4f} ± {np.std(valid_p):.4f}")
sig = np.sum(valid_p < 0.05)
print(f"  Significant (p<0.05): {sig}/{len(valid_p)} ({sig/len(valid_p)*100:.1f}%)")